# M11 — Interrogate a Decision Tree

Start from a trained shallow tree and explain how its learned splits create decisions. This lab is CPU-only, deterministic, local, and intentionally uses synthetic data.

## Mission contract

You will inspect thresholds, nodes, left branches, right branches, leaves, impurity, depth, minimum sample constraints, individual decision paths, and feature importance. For each consequential experiment, write a **Predict before running** note in your own evidence log before executing the action cell.

The target and all rows are synthetic. A path explains this fitted model's prediction for one row. It is not a causal explanation, a universal rule, or evidence for a real learner intervention.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 42
FEATURES = [
    "study_hours_week",
    "practice_accuracy",
    "attendance_pct",
    "sleep_hours",
]
TARGET = "ready_for_assessment"

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "datasets" / "M11" / "learner_readiness.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the LearningOS-AI repository.")

ROOT = find_repo_root()
pd.set_option("display.max_colwidth", 120)

In [ ]:
data_path = ROOT / "datasets" / "M11" / "learner_readiness.csv"
data = pd.read_csv(data_path)
assert list(data.columns) == ["learner_id", *FEATURES, TARGET]
assert len(data) == 96 and data["learner_id"].is_unique
assert set(data[TARGET]) == {0, 1}

X = data[FEATURES]
y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"rows={len(data)}, train={len(X_train)}, test={len(X_test)}")
print("target balance:", data[TARGET].value_counts().sort_index().to_dict())
data.head()

## Start from a trained shallow tree

**Predict before running:** With `max_depth=3` and `min_samples_leaf=4`, what is the longest possible number of split decisions? Will training accuracy necessarily be perfect? Record both predictions and your reasoning.

In [ ]:
shallow_tree = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=4, random_state=RANDOM_STATE
).fit(X_train, y_train)

def model_summary(model, name):
    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    test_accuracy = accuracy_score(y_test, model.predict(X_test))
    train_leaf_ids = model.apply(X_train)
    return {
        "model": name,
        "depth": model.get_depth(),
        "nodes": model.tree_.node_count,
        "leaves": model.get_n_leaves(),
        "smallest_train_leaf": int(pd.Series(train_leaf_ids).value_counts().min()),
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
        "generalization_gap": train_accuracy - test_accuracy,
    }

baseline_summary = pd.DataFrame([model_summary(shallow_tree, "depth=3, leaf>=4")])
baseline_summary.round(3)

## Read the learned structure

At an internal **node**, the fitted tree tests `feature <= threshold`. True follows the **left branch** and false follows the **right branch**. A **leaf** has no children and predicts from the class distribution of training rows that reached it. **Depth** counts split decisions from the root.

Gini **impurity** is low when a node mostly contains one class and high when classes are mixed. A candidate split is useful when the sample-weighted impurity of its children is lower than the parent's impurity.

In [ ]:
tree_text = export_text(shallow_tree, feature_names=FEATURES, decimals=3)
print(tree_text)

### Inspect nodes before trusting the picture

**Predict before running:** Which `tree_` fields must distinguish an internal node from a leaf? Pick one internal node from the text above and predict its feature, threshold, and children.

In [ ]:
def describe_nodes(model):
    tree = model.tree_
    rows = []
    for node_id in range(tree.node_count):
        left = int(tree.children_left[node_id])
        right = int(tree.children_right[node_id])
        is_leaf = left == right
        feature = None if is_leaf else FEATURES[int(tree.feature[node_id])]
        threshold = None if is_leaf else float(tree.threshold[node_id])
        distribution = np.asarray(tree.value[node_id]).reshape(-1)
        predicted_class = int(model.classes_[int(np.argmax(distribution))])
        rows.append({
            "node": node_id,
            "kind": "leaf" if is_leaf else "split",
            "feature": feature,
            "threshold": threshold,
            "left_child": None if is_leaf else left,
            "right_child": None if is_leaf else right,
            "samples": int(tree.n_node_samples[node_id]),
            "impurity": float(tree.impurity[node_id]),
            "class_distribution": np.round(distribution, 3).tolist(),
            "predicted_class": predicted_class,
        })
    return pd.DataFrame(rows)

node_table = describe_nodes(shallow_tree)
node_table.round(3)

In [ ]:
root = 0
left = shallow_tree.tree_.children_left[root]
right = shallow_tree.tree_.children_right[root]
root_samples = shallow_tree.tree_.n_node_samples[root]
weighted_child_impurity = (
    shallow_tree.tree_.n_node_samples[left] / root_samples * shallow_tree.tree_.impurity[left]
    + shallow_tree.tree_.n_node_samples[right] / root_samples * shallow_tree.tree_.impurity[right]
)
root_impurity = shallow_tree.tree_.impurity[root]
impurity_drop = root_impurity - weighted_child_impurity
assert impurity_drop >= -1e-12
pd.Series({
    "root_impurity": root_impurity,
    "weighted_child_impurity": weighted_child_impurity,
    "impurity_drop": impurity_drop,
}).round(4)

Explain the root calculation in plain language: how mixed was the parent, how mixed were the children after weighting by their sample counts, and how much mixing did the split remove? Do not describe impurity as error probability or causal impact.

## Experiment 1 — individual decision paths

**Predict before running:** Select the first two displayed test rows. Using the exported tree, write every comparison, predicted branch, and expected class for each row. Only then verify the paths.

In [ ]:
def explain_path(model, row):
    row_frame = row[FEATURES].to_frame().T
    indicator = model.decision_path(row_frame)
    node_ids = indicator.indices[indicator.indptr[0] : indicator.indptr[1]]
    leaf_id = int(model.apply(row_frame)[0])
    steps = []
    for node_id in node_ids:
        if node_id == leaf_id:
            distribution = np.asarray(model.tree_.value[node_id]).reshape(-1)
            steps.append({
                "node": int(node_id),
                "kind": "leaf",
                "decision": f"predict {int(model.classes_[np.argmax(distribution)])}",
            })
            continue
        feature = FEATURES[int(model.tree_.feature[node_id])]
        threshold = float(model.tree_.threshold[node_id])
        value = float(row[feature])
        go_left = value <= threshold
        steps.append({
            "node": int(node_id),
            "kind": "split",
            "decision": f"{feature}={value:.3f} {'<=' if go_left else '>'} {threshold:.3f} -> {'left' if go_left else 'right'}",
        })
    return pd.DataFrame(steps), leaf_id

selected_rows = X_test.iloc[:2]
for row_index, row in selected_rows.iterrows():
    path, leaf_id = explain_path(shallow_tree, row)
    print(f"row={data.loc[row_index, 'learner_id']}, actual={y_test.loc[row_index]}, leaf={leaf_id}")
    print(path.to_string(index=False), "\n")

Compare each manual path with the observed node sequence. If they differ, identify the first threshold comparison where your trace diverged. A final class without the comparisons is not path evidence.

## Experiment 2 — `max_depth` sweep

**Predict before running:** As `max_depth` rises while `min_samples_leaf=4` stays fixed, predict the directions of node count, training accuracy, test accuracy, and path length. Which quantity need not move monotonically?

In [ ]:
depth_models = {}
depth_rows = []
for max_depth in [1, 2, 3, 4, None]:
    model = DecisionTreeClassifier(
        max_depth=max_depth, min_samples_leaf=4, random_state=RANDOM_STATE
    ).fit(X_train, y_train)
    label = f"max_depth={max_depth}"
    depth_models[label] = model
    depth_rows.append(model_summary(model, label))
depth_results = pd.DataFrame(depth_rows)
depth_results.round(3)

In [ ]:
ax = depth_results.plot(
    x="depth", y=["train_accuracy", "test_accuracy"], marker="o", ylim=(0.5, 1.02)
)
ax.set_title("Depth changes fit and held-out behavior differently")
ax.set_ylabel("accuracy")
ax.grid(alpha=0.25)
plt.show()

## Experiment 3 — shallow versus deep, train versus test

**Predict before running:** An unconstrained tree can isolate noisy examples. Predict its training accuracy, smallest leaf, and train–test gap relative to the shallow baseline. State what held-out result would weaken your overfitting hypothesis.

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
comparison = pd.DataFrame([
    model_summary(shallow_tree, "shallow: depth=3, leaf>=4"),
    model_summary(deep_tree, "deep: unconstrained"),
])
assert comparison.loc[1, "train_accuracy"] >= comparison.loc[0, "train_accuracy"]
comparison.round(3)

Diagnose from the whole row of evidence. More nodes or greater depth alone do not prove overfitting; the seductive training fit, tiny leaves, and worse held-out behavior together make the case. A test set is evidence about this split, not a guarantee about future populations.

## Experiment 4 — perturb one feature

**Predict before running:** The next cell selects a row near the root threshold and creates values just below and above that threshold while holding every other feature fixed. Predict the first changed branch. Does a changed branch guarantee a changed final class?

In [ ]:
root_feature = FEATURES[int(shallow_tree.tree_.feature[0])]
root_threshold = float(shallow_tree.tree_.threshold[0])
nearest_position = int(np.argmin(np.abs(X_test[root_feature].to_numpy() - root_threshold)))
original = X_test.iloc[nearest_position].copy()
feature_span = float(X[root_feature].max() - X[root_feature].min())
epsilon = max(feature_span * 0.02, 0.001)
below = original.copy()
above = original.copy()
below[root_feature] = root_threshold - epsilon
above[root_feature] = root_threshold + epsilon

perturbation_rows = pd.DataFrame([original, below, above], index=["original", "below_root", "above_root"])
perturbation_rows["prediction"] = shallow_tree.predict(perturbation_rows[FEATURES])
perturbation_rows["leaf"] = shallow_tree.apply(perturbation_rows[FEATURES])
print(f"root test: {root_feature} <= {root_threshold:.3f}")
display(perturbation_rows)
for label in ["below_root", "above_root"]:
    path, _ = explain_path(shallow_tree, perturbation_rows.loc[label, FEATURES])
    print(label, "\n", path.to_string(index=False), "\n")

This is a counterfactual **to the fitted model**: it reveals model sensitivity while other recorded values are held fixed. It is not evidence that intervening on the feature would cause the real outcome. Record whether the root branch, leaf, and prediction changed; these are three distinct observations.

## Experiment 5 — `min_samples_leaf`

**Predict before running:** With `max_depth=None`, predict how increasing `min_samples_leaf` affects the smallest leaf, number of leaves, training accuracy, and test accuracy. Which outcomes are mechanically constrained, and which remain empirical?

In [ ]:
leaf_rows = []
leaf_models = {}
for min_samples_leaf in [1, 2, 4, 8, 12]:
    model = DecisionTreeClassifier(
        max_depth=None, min_samples_leaf=min_samples_leaf, random_state=RANDOM_STATE
    ).fit(X_train, y_train)
    label = f"min_samples_leaf={min_samples_leaf}"
    leaf_models[label] = model
    leaf_rows.append(model_summary(model, label))
leaf_results = pd.DataFrame(leaf_rows)
leaf_results.round(3)

## Feature importance, cautiously

**Predict before running:** Rank the four features from the exported shallow tree. Then predict one reason impurity-based and held-out permutation importance might disagree.

Impurity-based importance totals how much this fitted tree used a feature to reduce weighted impurity. It can favor features with more candidate splits, divide credit unpredictably among correlated features, and change with a new sample or tree. Permutation importance asks how held-out score changes after shuffling a column, but it too is model- and dataset-specific. Neither measure is a causal effect.

In [ ]:
permutation = permutation_importance(
    shallow_tree, X_test, y_test, scoring="accuracy", n_repeats=30, random_state=RANDOM_STATE
)
importance_comparison = pd.DataFrame({
    "feature": FEATURES,
    "impurity_importance": shallow_tree.feature_importances_,
    "test_permutation_mean": permutation.importances_mean,
    "test_permutation_std": permutation.importances_std,
}).sort_values("impurity_importance", ascending=False)
importance_comparison.round(3)

## Controlled failure — repair the over-deep tree

**Predict before running:** Fit a repair with both a depth cap and larger leaves. Predict which metrics must decrease, which may improve, and what result would show that the chosen repair is not better on this held-out split.

In [ ]:
repaired_tree = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=6, random_state=RANDOM_STATE
).fit(X_train, y_train)
failure_comparison = pd.DataFrame([
    model_summary(deep_tree, "failure: unconstrained"),
    model_summary(repaired_tree, "repair: depth=4, leaf>=6"),
])
generalization_gap = dict(zip(failure_comparison["model"], failure_comparison["generalization_gap"]))
assert failure_comparison.loc[0, "smallest_train_leaf"] <= failure_comparison.loc[1, "smallest_train_leaf"]
failure_comparison.round(3)

### Controlled failure — reject causal overreach

Deliberately invalid claim: “The top feature importance means increasing that feature will cause readiness.”

Before continuing, write: (1) your verdict, (2) the causal evidence that is missing, and (3) a safer model-specific sentence. Your sentence may describe which feature this fitted tree used for predictive partitions; it may not promise that an intervention changes the target.

## ADR prompt

Complete `missions/M11/adr_prompt.md`. Choose a constrained baseline or “do not deploy” using the unchanged test evidence, path length, leaf sizes, interpretation risks, trade-offs, and revisit conditions. The only admissible acceptance status in this mission is for offline V03 comparison.

## No-AI gate

Complete `missions/M11/no_ai_gate.md` without AI-generated code or prose. Trace both fresh rows, compare a shallow and an unconstrained tree using train/test evidence, predict the effect of `min_samples_leaf`, and reject the causal claim.

## Completion check

Your evidence should now contain: shallow baseline; node and impurity explanation; two individual paths; `max_depth` sweep; shallow/deep train/test comparison; one-feature perturbation; `min_samples_leaf` sweep; cautious importance comparison; controlled-failure repair; causal-claim audit; ADR; and fresh no-AI transfer.

Restart the kernel and Run All. A successful execution proves reproducibility of the artifact, not mastery; mastery requires your predictions and explanations.